# Has-value and added-value figures (NREM and Wake)

Assembles the publication panels from data exported by the companion notebooks
(`incremental_added_value.ipynb` for NREM, `incremental_added_value_wake.ipynb` for
Wake), which write their precomputed coefficients and epochs to
`./outputs/added_value_data/`. This notebook only plots: no fitting, no NFS, so the
figures and the analyses share one source of truth.

Each panel is its own cell and its own figure, saved separately, to be assembled later
in the vector editor. There are three panels per state:

- Has value vs adds value (dumbbell): each tier's marginal coefficient (alone, upper,
  open) against its partial coefficient (holding the other two tiers fixed, lower,
  filled), on offset rows so each interval attaches to its own estimate. The 95% CI is a
  thick band with a small dot at the point estimate rather than a whisker through a
  large marker: at 1 inch the pooled CIs (half-width 0.03-0.09) are narrower than a
  conventional 6 pt marker (0.11 data units across in the NREM panel), which hid most of
  them. 1 x 1 inch.
- Added value across recordings (forest): per-recording partial coefficients, with
  CLAS-exclusive and LLAS-exclusive in side-by-side subplots on a shared x-axis, each
  sorted into its own caterpillar with the random-effects pooled diamond and value. Laid
  out side by side (double width), 1.5 inches tall. A three-panel variant additionally
  shows BLAS-exclusive, i.e. BLAS itself, as a left panel. Rows within a subplot are
  independently sorted, so a given height is not a shared recording.
- Model-light cross-check (stratified): mean delta against a focus tier's OFF area by
  quantile, within strata of the bigger OFFs. Color is the focus tier, line style the
  stratum. 1 inch tall.

NREM uses the continuous area model, Wake the occurrence (presence) model.

Styling: plotted under `pubplots.destination("figma")`. Font sizes are left entirely to
pubplots, uniform across panels. Marker sizes and line weights go through `pp.scale`.
The palette matches `has_value.ipynb`, the shared Set2 category palette: LLAS-exclusive
green, CLAS-exclusive orange, BLAS blue. Panels carry no titles. When
`PLOT_FOR_PUB = True` all axis furniture (tick labels, legends, axis labels) is stripped
so the bare panels can be assembled and annotated in the vector editor; set it `False`
to restore the labeled quick-look versions. Saved as SVG, the deliverable, and PNG for
quick on-screen inspection.

Reading the partial coefficient, e.g. BLAS in NREM at +0.68: the expected change in
epoch delta (SD units) per 1 SD more of that tier's OFF area, with the other tiers held
fixed, which is that tier's unique, non-redundant contribution. For BLAS, marginal and
partial are about equal; for a suppressor like CLAS-exclusive the partial exceeds the
marginal.


In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pubplots as pp
from matplotlib.lines import Line2D

from cnpix_local_sleep import plots

DATA = pathlib.Path("./outputs/added_value_data")
OUT = pathlib.Path("./outputs/added_value_figures")
OUT.mkdir(parents=True, exist_ok=True)
save_plots = True

# Publication toggle: when True, strip all axis furniture (tick labels, legends,
# axis labels) so the bare panels can be assembled/annotated in the vector editor.
PLOT_FOR_PUB = True

# Annotate the partial (filled) markers of the dumbbell panels with the squared
# semipartial = that level's increment in R2 over the other level. It is a magnitude;
# direction is carried by the marker position. Survives PLOT_FOR_PUB because it is
# data, not axis furniture -- keeping it in code stops it drifting from the parquets.
ANNOTATE_DR2 = True

# Palette matches has_value.ipynb: the shared Set2 category palette
# (plots.get_category_palette), so each added-value tier reuses
# its parent LAS category's color -- LLAS-exclusive the LLAS green, CLAS-exclusive
# the CLAS orange, BLAS the BLAS blue. The Wake amount model collapses BLAS +
# CLAS-exclusive into one "Conservative (CLAS set)" predictor; it takes the
# CLAS-exclusive orange (the CLAS set is CLAS-exclusive + BLAS, so the orange
# reads as "the CLAS tier").
_CAT = plots.get_category_palette()
TIER_COLORS = {"BLAS": _CAT["BLAS"], "CLAS-exclusive": _CAT["CLAS"],
               "LLAS-exclusive": _CAT["LLAS"],
               "Conservative (CLAS set)": _CAT["CLAS"]}
ORDER = ["BLAS", "CLAS-exclusive", "LLAS-exclusive"]
AREA_ORDER = ["Conservative (CLAS set)", "LLAS-exclusive"]  # Wake amount model (2 predictors)


# The reported partial quantity is the SEMIPARTIAL (part) coefficient
# beta_sr = beta_joint * sqrt(1 - R2_i), not the joint coefficient. Its square is the
# tier's incremental R2 (R2(all) - R2(others)), so a marginal/partial pair reads
# simultaneously as coefficients and as marginal-vs-incremental variance explained --
# while keeping the sign, which R2 alone discards. The joint coefficients are still on
# disk in *_partial_pooled.parquet if a sensitivity check needs them.
def _use_semipartial(groups):
    """Point the plain beta_/se_ columns at their _sr variants, so every per-group
    panel reports the same partial quantity without changing the plotting calls."""
    g = groups.copy()
    for c in [c for c in g.columns if c.endswith("_sr")]:
        g[c[: -len("_sr")]] = g[c]
    return g


def load(prefix):
    return dict(
        partial=pd.read_parquet(DATA / f"{prefix}_semipartial_pooled.parquet"),
        joint=pd.read_parquet(DATA / f"{prefix}_partial_pooled.parquet"),
        marginal=pd.read_parquet(DATA / f"{prefix}_marginal_pooled.parquet"),
        groups=_use_semipartial(pd.read_parquet(DATA / f"{prefix}_group_partial.parquet")),
        epochs=pd.read_parquet(DATA / f"{prefix}_strat_epochs.parquet"),
    )


def load_area(prefix):
    """Amount (area) model: two estimable tiers, no stratified-epoch export."""
    return dict(
        partial=pd.read_parquet(DATA / f"{prefix}_semipartial_pooled.parquet"),
        joint=pd.read_parquet(DATA / f"{prefix}_partial_pooled.parquet"),
        marginal=pd.read_parquet(DATA / f"{prefix}_marginal_pooled.parquet"),
        groups=_use_semipartial(pd.read_parquet(DATA / f"{prefix}_group_partial.parquet")),
    )


nrem = load("nrem")
wake = load("wake")
wake_area = load_area("wake_area")
nrem_area = load_area("nrem_area")
for name, d in [("NREM (3-tier)", nrem), ("Wake (presence)", wake),
                ("NREM amount", nrem_area), ("Wake amount", wake_area)]:
    t = d["partial"][["tier", "pooled_std_beta", "ci_lo", "ci_hi"]].copy()
    t["dR2"] = t["pooled_std_beta"] ** 2  # display only -- NOT a pooled dR2
    print(f"{name} semipartial:\n", t.to_string(index=False), "\n")

## Panel functions

Font sizes come from the figma destination. Marker sizes and line weights use
`pp.scale`; data-coordinate positions (row offsets, diamond geometry) are not scaled.
Legends that would overlap the data are placed outside the axes, with explicit figure
margins reserving room for them.


In [ ]:
# Dumbbell glyph geometry. The 95% CI is drawn as a thick band and the point estimate
# as a small dot on top of it -- not as an errorbar whisker through a large marker.
# At a 1-inch panel the x-span is set by the spread of the point estimates
# (NREM: BLAS +0.73 vs LLAS-exclusive -0.29, so 1.25 units across ~0.92 inches of axes)
# while the pooled CI half-widths are only 0.027-0.088. A 6 pt marker is 0.113 data
# units across, so it swallowed 8 of the 22 whiskers entirely (worst: NREM BLAS
# partial, half-width 0.48x the marker radius). The band inherits the marker's visual
# weight in y (thickness, which is free -- y is categorical) and spends x purely on the
# interval, so no CI can hide. The dot is kept below the narrowest half-width (0.027).
BAND_LW = 4.5  # CI band thickness, pt (pre-pp.scale)
DOT_MS = 2.4  # point-estimate dot, pt (pre-pp.scale)


def _ci_glyph(ax, row, y, c, filled):
    """Draw one estimate: CI band + point-estimate dot, centered on row ``y``.

    ``solid_capstyle="butt"`` is required -- the default round cap would overhang
    each end by half the linewidth, inflating the interval by ~0.05 data units.
    ``filled`` keeps the open/filled language of the previous markers on the dot
    (partial = solid tier-colored, marginal = white with a tier edge); the band is
    translucent in both rows so the dot always reads against it.
    """
    ax.plot([row["ci_lo"], row["ci_hi"]], [y, y], color=c, lw=pp.scale(BAND_LW),
            solid_capstyle="butt", alpha=0.55 if filled else 0.35, zorder=2)
    ax.plot([row["pooled_std_beta"]], [y], marker="o", ls="",
            mfc=c if filled else "white", mec=c,
            ms=pp.scale(DOT_MS), mew=pp.scale(0.6), zorder=3)


def panel_dumbbell(ax, marg, part, xlabel, xlim=None, order=None):
    """Marginal (open, upper row) vs partial (filled, lower row) per tier.

    Each estimate is a thick 95% CI band with a small dot at the point estimate (see
    ``_ci_glyph``), on offset rows so each interval attaches to its own estimate.
    ``order`` selects which tiers (and in what top-to-bottom order) to draw;
    defaults to the 3-tier NREM/Wake ORDER. The Wake amount model passes
    AREA_ORDER (2 tiers). No title. When ``PLOT_FOR_PUB`` the y-tick labels,
    legend, and x-label are suppressed.
    """
    order = list(order) if order is not None else ORDER
    dy = 0.18  # data-coordinate row offset (not scaled)
    # Span of everything that will be drawn. Needed up front because the axes are not
    # autoscaled until after the loop, so ax.get_xlim() here would be meaningless.
    _b = [f[f["tier"] == t].iloc[0] for f in (marg, part) for t in order]
    span_lo, span_hi = min(r["ci_lo"] for r in _b), max(r["ci_hi"] for r in _b)
    if xlim is not None:
        span_lo, span_hi = xlim
    for i, tier in enumerate(order):
        yy = len(order) - 1 - i
        m = marg[marg["tier"] == tier].iloc[0]
        p = part[part["tier"] == tier].iloc[0]
        c = TIER_COLORS[tier]
        ax.plot([m["pooled_std_beta"], p["pooled_std_beta"]], [yy + dy, yy - dy],
                color=c, lw=pp.scale(1.2), alpha=0.45, zorder=1)
        _ci_glyph(ax, m, yy + dy, c, filled=False)
        _ci_glyph(ax, p, yy - dy, c, filled=True)
        if ANNOTATE_DR2:
            # Park the value in whichever side of this row is free, at the row's own y
            # (x in axes fraction, y in data coords). Bare number: the caption names it,
            # and "ΔR²=" does not fit a 1-inch panel.
            row_lo = min(m["ci_lo"], p["ci_lo"])
            row_hi = max(m["ci_hi"], p["ci_hi"])
            right = (span_hi - row_hi) >= (row_lo - span_lo)
            ax.text(0.97 if right else 0.03, yy - dy, f"{p['pooled_std_beta'] ** 2:.2f}",
                    transform=ax.get_yaxis_transform(), ha="right" if right else "left",
                    va="center", color=c, zorder=4)
    ax.axvline(0, color="grey", ls="--", lw=pp.scale(0.8), zorder=0)
    ax.set_yticks(range(len(order)))
    ax.set_ylim(-0.6, len(order) - 0.4)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if PLOT_FOR_PUB:
        ax.set_yticklabels([])
        return
    ax.set_yticklabels(order[::-1])
    ax.set_xlabel(xlabel)
    # legend outside the axes (right) so it never sits on a tier's markers
    _h = dict(ls="-", lw=pp.scale(BAND_LW), marker="o", ms=pp.scale(DOT_MS), mew=pp.scale(0.6))
    ax.legend(handles=[Line2D([0], [0], color="k", alpha=0.35, mfc="white", mec="k",
                              label="marginal", **_h),
                       Line2D([0], [0], color="k", alpha=0.55, mfc="k", mec="k",
                              label="partial", **_h)],
              loc="center left", bbox_to_anchor=(1.03, 0.5), handletextpad=0.4, borderaxespad=0.0)


def forest_subplot(ax, gdf, name, bcol, scol, pooled, show_xlabel=False, xlabel=""):
    """One tier's sorted caterpillar + pooled diamond, for a shared-x forest.
    Tier name goes on the y-axis (points are tier-colored); the pooled value is
    annotated in the empty lower-right corner. When ``PLOT_FOR_PUB`` the axis
    labels are suppressed."""
    c = TIER_COLORS[name]
    d = gdf.dropna(subset=[bcol, scol]).sort_values(bcol).reset_index(drop=True)
    k = len(d)
    y = np.arange(k)
    pr = pooled[pooled["tier"] == name].iloc[0]
    ax.errorbar(d[bcol], y, xerr=1.96 * d[scol], fmt="o", markersize=pp.scale(3),
                color=c, ecolor=c, elinewidth=pp.scale(0.6), capsize=0, alpha=0.8, zorder=3)
    yd, hw = -3.2, 2.0  # diamond center / half-height (data coords)
    ax.fill([pr["ci_lo"], pr["pooled_std_beta"], pr["ci_hi"], pr["pooled_std_beta"]],
            [yd, yd + hw, yd, yd - hw], color=c, alpha=0.9, zorder=5)
    ax.axvline(0, color="grey", ls="--", lw=pp.scale(0.8), zorder=0)
    ax.set_yticks([])
    ax.set_ylim(yd - hw - 0.8, k + 1)
    ax.text(0.975, 0.05, f"pooled {pr['pooled_std_beta']:+.2f}", transform=ax.transAxes,
            ha="right", va="bottom", color=c)
    if PLOT_FOR_PUB:
        return
    ax.set_ylabel(name)
    if show_xlabel:
        ax.set_xlabel(xlabel)


def panel_stratified(ax, ep, focuses, strata_order, styles, strat_title, n_q=5, skip=()):
    """Color = focus tier; line style = stratum. skip: (tier, stratum) pairs not to
    plot (focus tier identically zero there by construction). No title. When
    ``PLOT_FOR_PUB`` the axis labels and legend are suppressed; otherwise one
    combined legend (tier group + stratum sub-header) is placed outside the axes."""
    skip = set(skip)
    for name, focus_col in focuses:
        c = TIER_COLORS[name]
        dd = ep.copy()
        dd["q"] = dd.groupby("strat", observed=True)[focus_col].transform(
            lambda s: pd.qcut(s.rank(method="first"), n_q, labels=False))
        agg = dd.groupby(["strat", "q"], observed=True)["mean_log_delta"].agg(["mean", "sem"]).reset_index()
        for st, ls in zip(strata_order, styles):
            if (name, st) in skip:
                continue
            sub = agg[agg["strat"] == st]
            ax.errorbar(sub["q"], sub["mean"], yerr=sub["sem"], color=c, ls=ls,
                        marker="o", ms=pp.scale(4), capsize=pp.scale(1.5),
                        elinewidth=pp.scale(0.8), lw=pp.scale(1.3))
    ax.set_xticks(range(n_q))
    if PLOT_FOR_PUB:
        ax.set_xticklabels([])
        return
    ax.set_xlabel("OFF-area quantile (within stratum)")
    ax.set_ylabel("mean z(log δ)")
    color_h = [Line2D([0], [0], color=TIER_COLORS[n], marker="o", ls="-", label=n) for n, _ in focuses]
    style_h = [Line2D([0], [0], color="grey", ls=ls, label=lab) for lab, ls in zip(strata_order, styles)]
    blank = Line2D([0], [0], ls="none", label="")
    subhdr = Line2D([0], [0], ls="none", label=strat_title)
    ax.legend(handles=color_h + [blank, subhdr] + style_h, loc="center left",
              bbox_to_anchor=(1.03, 0.5), title="focus tier", handletextpad=0.6, borderaxespad=0.0)


def save_fig(fig, stem):
    """SVG is the deliverable; PNG is a quick-look raster saved alongside it."""
    if save_plots:
        fig.savefig(OUT / f"{stem}.svg")
        fig.savefig(OUT / f"{stem}.png", dpi=200)

## NREM: has value vs adds value (dumbbell)

In [ ]:
with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale((1, 1)))
    panel_dumbbell(ax, nrem["marginal"], nrem["partial"],
                   "correlation with delta power (OFF area)", xlim=(-0.43, 0.82))
    if PLOT_FOR_PUB:
        fig.subplots_adjust(left=0.06, right=0.98, top=0.98, bottom=0.06)
    else:
        fig.subplots_adjust(left=0.28, right=0.62, top=0.95, bottom=0.20)
    save_fig(fig, "figure_nrem_dumbbell")
    plt.show()

## NREM: added value across recordings (forest, side-by-side)

In [ ]:
with pp.destination("figma"):
    fig, (axl, axr) = plt.subplots(1, 2, figsize=pp.scale((3.0, 1.5)), sharex=True, constrained_layout=True)
    forest_subplot(axl, nrem["groups"], "CLAS-exclusive", "beta_clas_excl", "se_clas_excl", nrem["partial"],
                   show_xlabel=True, xlabel="semipartial correlation")
    forest_subplot(axr, nrem["groups"], "LLAS-exclusive", "beta_llas_excl", "se_llas_excl", nrem["partial"],
                   show_xlabel=True, xlabel="semipartial correlation")
    save_fig(fig, "figure_nrem_forest")
    plt.show()

## NREM: added value across recordings (forest, 3 panels: BLAS + CLAS-exclusive + LLAS-exclusive)

In [ ]:
# Three-panel variant: adds a BLAS-exclusive panel. BLAS is already the most
# conservative tier, so "BLAS-exclusive" is just BLAS itself (nothing coarser to
# subtract), drawn from the same partial fit (beta_blas / se_blas).
with pp.destination("figma"):
    fig, (axl, axm, axr) = plt.subplots(1, 3, figsize=pp.scale((4.5, 1.5)), sharex=True, constrained_layout=True)
    forest_subplot(axl, nrem["groups"], "BLAS", "beta_blas", "se_blas", nrem["partial"],
                   show_xlabel=True, xlabel="semipartial correlation")
    forest_subplot(axm, nrem["groups"], "CLAS-exclusive", "beta_clas_excl", "se_clas_excl", nrem["partial"],
                   show_xlabel=True, xlabel="semipartial correlation")
    forest_subplot(axr, nrem["groups"], "LLAS-exclusive", "beta_llas_excl", "se_llas_excl", nrem["partial"],
                   show_xlabel=True, xlabel="semipartial correlation")
    save_fig(fig, "figure_nrem_forest3")
    plt.show()

## NREM: model-light cross-check (stratified)

In [ ]:
ep = nrem["epochs"].copy()
ep["strat"] = pd.qcut(ep["blas_area"].rank(method="first"), 3,
                      labels=["BLAS low", "BLAS mid", "BLAS high"])

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale((2.0, 1.0)))
    panel_stratified(ax, ep,
                     [("LLAS-exclusive", "llas_excl_area"), ("CLAS-exclusive", "clas_excl_area")],
                     ["BLAS low", "BLAS mid", "BLAS high"], ["dotted", "dashed", "solid"],
                     "BLAS stratum")
    if PLOT_FOR_PUB:
        fig.subplots_adjust(left=0.06, right=0.98, top=0.97, bottom=0.08)
    else:
        fig.subplots_adjust(left=0.16, right=0.62, top=0.95, bottom=0.20)
    save_fig(fig, "figure_nrem_stratified")
    plt.show()

## Wake: has value vs adds value (dumbbell, presence)

Wake panels adapt to the occurrence (presence) model and Wake's sparsity. All three
tiers are positive, so the dumbbell x-limits open a clear pocket for the legend; the
cross-check is stratified by conservative-OFF presence, BLAS being too rare in Wake to
stratify. The CLAS-exclusive curve in the no-conservative-OFF stratum is omitted, since
`conservative = BLAS + CLAS-exclusive` makes CLAS-exclusive identically zero there, a
z-scoring artifact. The remaining CLAS-exclusive curve is sparse, so read it
qualitatively.


In [ ]:
with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale((1, 1)))
    panel_dumbbell(ax, wake["marginal"], wake["partial"],
                   "correlation with delta power (OFF presence)", xlim=(-0.05, 0.62))
    if PLOT_FOR_PUB:
        fig.subplots_adjust(left=0.06, right=0.98, top=0.98, bottom=0.06)
    else:
        fig.subplots_adjust(left=0.28, right=0.62, top=0.95, bottom=0.20)
    save_fig(fig, "figure_wake_dumbbell")
    plt.show()

## Wake: added value across recordings (forest, side-by-side)

In [ ]:
with pp.destination("figma"):
    fig, (axl, axr) = plt.subplots(1, 2, figsize=pp.scale((3.0, 1.5)), sharex=True, constrained_layout=True)
    forest_subplot(axl, wake["groups"], "CLAS-exclusive", "beta_clas_excl_any", "se_clas_excl_any", wake["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (presence)")
    forest_subplot(axr, wake["groups"], "LLAS-exclusive", "beta_llas_excl_any", "se_llas_excl_any", wake["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (presence)")
    save_fig(fig, "figure_wake_forest")
    plt.show()

## Wake: added value across recordings (forest, 3 panels: BLAS + CLAS-exclusive + LLAS-exclusive)

In [ ]:
# Three-panel variant: adds a BLAS-exclusive panel. BLAS is already the most
# conservative tier, so "BLAS-exclusive" is just BLAS itself (nothing coarser to
# subtract), drawn from the same presence fit (beta_blas_any / se_blas_any).
with pp.destination("figma"):
    fig, (axl, axm, axr) = plt.subplots(1, 3, figsize=pp.scale((4.5, 1.5)), sharex=True, constrained_layout=True)
    forest_subplot(axl, wake["groups"], "BLAS", "beta_blas_any", "se_blas_any", wake["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (presence)")
    forest_subplot(axm, wake["groups"], "CLAS-exclusive", "beta_clas_excl_any", "se_clas_excl_any", wake["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (presence)")
    forest_subplot(axr, wake["groups"], "LLAS-exclusive", "beta_llas_excl_any", "se_llas_excl_any", wake["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (presence)")
    save_fig(fig, "figure_wake_forest3")
    plt.show()

## Wake: model-light cross-check (stratified)

In [ ]:
epw = wake["epochs"].copy()
epw["strat"] = epw["cons_any"].map({0.0: "no conservative OFF", 1.0: "has conservative OFF"})

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale((2.0, 1.0)))
    panel_stratified(ax, epw,
                     [("LLAS-exclusive", "llas_excl_area"), ("CLAS-exclusive", "clas_excl_area")],
                     ["no conservative OFF", "has conservative OFF"], ["dotted", "solid"],
                     "conservative OFF",
                     skip={("CLAS-exclusive", "no conservative OFF")})
    if PLOT_FOR_PUB:
        fig.subplots_adjust(left=0.06, right=0.98, top=0.97, bottom=0.08)
    else:
        fig.subplots_adjust(left=0.16, right=0.62, top=0.95, bottom=0.20)
    save_fig(fig, "figure_wake_stratified")
    plt.show()

## Wake: area model (secondary), has value vs adds value (dumbbell)

The amount (collapsed) companion to the Wake occurrence dumbbell above. Only two tiers
are estimable: Conservative (CLAS set) = BLAS + CLAS-exclusive, and LLAS-exclusive,
because epoch-scale BLAS area is near-degenerate in Wake and is folded into the
conservative set. Marginal (open) is each tier's amount coefficient alone; partial
(filled) holds the other fixed.


In [ ]:
with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale((1, 1)))
    # xlim widened past the data to reserve a gutter for the dR2 annotations.
    panel_dumbbell(ax, wake_area["marginal"], wake_area["partial"],
                   "correlation with delta power (OFF area)", order=AREA_ORDER,
                   xlim=(-0.04, 0.42))
    if PLOT_FOR_PUB:
        fig.subplots_adjust(left=0.06, right=0.98, top=0.98, bottom=0.06)
    else:
        fig.subplots_adjust(left=0.40, right=0.62, top=0.95, bottom=0.24)
    save_fig(fig, "figure_wake_area_dumbbell")
    plt.show()

## Wake: area model (secondary), added value across recordings (forest, side-by-side)

Per-recording partial amount coefficients from the collapsed model, with Conservative
(CLAS set) and LLAS-exclusive in side-by-side subplots on a shared x-axis, each sorted
into its own caterpillar with the random-effects pooled diamond and value. Rows within a
subplot are independently sorted, so a given height is not a shared recording. There are
two tiers rather than three because epoch-scale BLAS area is near-degenerate in Wake and
is folded into the conservative set.


In [ ]:
with pp.destination("figma"):
    fig, (axl, axr) = plt.subplots(1, 2, figsize=pp.scale((3.0, 1.5)), sharex=True, constrained_layout=True)
    forest_subplot(axl, wake_area["groups"], "Conservative (CLAS set)",
                   "beta_cons_area", "se_cons_area", wake_area["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (OFF area)")
    forest_subplot(axr, wake_area["groups"], "LLAS-exclusive",
                   "beta_llas_excl_area", "se_llas_excl_area", wake_area["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (OFF area)")
    save_fig(fig, "figure_wake_area_forest")
    plt.show()

# NREM: two-tier versions (CLAS and LLAS-exclusive only)

Two distinct NREM analogs of the Wake amount panels, kept separate because they come
from different models:

1. Subset of the 3-tier fit: the same `nrem_*` fit as above, displaying CLAS-exclusive
   and LLAS-exclusive with BLAS dropped. Each coefficient still holds all three tiers
   fixed, so this is a display subset and not a refit. Here "CLAS" means CLAS-exclusive,
   the increment.
2. Collapsed amount (area) model: a genuinely different 2-predictor fit (`nrem_area_*`,
   produced by `incremental_added_value.ipynb`) where Conservative (CLAS set) = BLAS +
   CLAS-exclusive is fit jointly with LLAS-exclusive, the exact structural analog of the
   Wake amount model. Here "CLAS" means the whole conservative CLAS set.

Same styling, sizes and `PLOT_FOR_PUB` behavior as the panels above.


## NREM (subset of 3-tier fit): has value vs adds value (dumbbell)

In [ ]:
# Subset of the existing 3-tier NREM fit: drop BLAS, show CLAS-exclusive +
# LLAS-exclusive. Coefficients are unchanged (still hold all three tiers fixed).
NREM_TWO_TIER = ["CLAS-exclusive", "LLAS-exclusive"]
with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale((1, 1)))
    panel_dumbbell(ax, nrem["marginal"], nrem["partial"],
                   "correlation with delta power (OFF area)", order=NREM_TWO_TIER)
    if PLOT_FOR_PUB:
        fig.subplots_adjust(left=0.06, right=0.98, top=0.98, bottom=0.06)
    else:
        fig.subplots_adjust(left=0.30, right=0.62, top=0.95, bottom=0.20)
    save_fig(fig, "figure_nrem_two_tier_dumbbell")
    plt.show()

## NREM (subset of 3-tier fit): added value across recordings (forest, side-by-side)

CLAS-exclusive and LLAS-exclusive per-recording partials from the 3-tier fit. This is
the same content as the primary NREM forest above, minus the BLAS panel.

In [ ]:
with pp.destination("figma"):
    fig, (axl, axr) = plt.subplots(1, 2, figsize=pp.scale((3.0, 1.5)), sharex=True, constrained_layout=True)
    forest_subplot(axl, nrem["groups"], "CLAS-exclusive", "beta_clas_excl", "se_clas_excl", nrem["partial"],
                   show_xlabel=True, xlabel="semipartial correlation")
    forest_subplot(axr, nrem["groups"], "LLAS-exclusive", "beta_llas_excl", "se_llas_excl", nrem["partial"],
                   show_xlabel=True, xlabel="semipartial correlation")
    save_fig(fig, "figure_nrem_two_tier_forest")
    plt.show()

## NREM amount (collapsed): has value vs adds value (dumbbell)

The true Wake-amount analog: a separately fit 2-predictor model (`nrem_area_*`) with
Conservative (CLAS set) = BLAS + CLAS-exclusive and LLAS-exclusive. Marginal (open) is
each tier's amount coefficient alone; partial (filled) holds the other fixed.


In [ ]:
# Collapsed NREM amount model (mirrors wake_area). Same 2 tiers/labels as AREA_ORDER.
# (loaded in the setup cell alongside the other three models)
print("NREM amount partial:\n",
      nrem_area["partial"][["tier", "pooled_std_beta", "ci_lo", "ci_hi"]].to_string(index=False))

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale((1, 1)))
    panel_dumbbell(ax, nrem_area["marginal"], nrem_area["partial"],
                   "correlation with delta power (OFF area)", order=AREA_ORDER)
    if PLOT_FOR_PUB:
        fig.subplots_adjust(left=0.06, right=0.98, top=0.98, bottom=0.06)
    else:
        fig.subplots_adjust(left=0.40, right=0.62, top=0.95, bottom=0.24)
    save_fig(fig, "figure_nrem_area_dumbbell")
    plt.show()

## NREM amount (collapsed): added value across recordings (forest, side-by-side)

Per-recording partial amount coefficients from the collapsed model, with Conservative
(CLAS set) and LLAS-exclusive in side-by-side subplots on a shared x-axis, each sorted
into its own caterpillar with the random-effects pooled diamond and value.


In [ ]:
with pp.destination("figma"):
    fig, (axl, axr) = plt.subplots(1, 2, figsize=pp.scale((3.0, 1.5)), sharex=True, constrained_layout=True)
    forest_subplot(axl, nrem_area["groups"], "Conservative (CLAS set)",
                   "beta_cons_area", "se_cons_area", nrem_area["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (OFF area)")
    forest_subplot(axr, nrem_area["groups"], "LLAS-exclusive",
                   "beta_llas_excl_area", "se_llas_excl_area", nrem_area["partial"],
                   show_xlabel=True, xlabel="semipartial correlation (OFF area)")
    save_fig(fig, "figure_nrem_area_forest")
    plt.show()